# Wave 2 Multivariate Linear Regression: Personal + Peer Influence

## Objective
To analyze the combined influence of **Personal Factors (Top 20)** and **Peer Depression** on **Own Depression** in Wave 2.

## Variables (21 Independent Variables)
1.  **Peer Factor**:
    *   `w2_peer_avg`: Average depression score of friends (Standardized).
2.  **Personal Factors (Top 20 from Correlation Analysis)**:
    *   `v51`: Happiness
    *   `v59_5`: Sleep Quality
    *   `v52_3`: Self-Satisfaction
    *   `v57_4`: Waking up fresh
    *   `v52_2`: Optimism
    *   `v57_3`: Energy
    *   `v50`: Life Satisfaction
    *   `v57_2`: Calmness
    *   `v52`: Health Status
    *   `v57_1`: Cheerful Spirit
    *   `v52_1`: Self-Worth
    *   `v57_5`: Interest in Life
    *   `v24_2`: Fear of Criticism (Social Media)
    *   `v8_08`: Study Burden
    *   `v28_6`: Excessive Internet Use
    *   `v39_2`: Cyberbullying (Victim)
    *   `v5_5`: Family Comfort
    *   `v8_05`: Being Misunderstood
    *   `v24_6`: Anxiety before posting
    *   `v35_2`: Real-world Bullying (Victim)

## Methodology
1.  **Standardization**: All variables (X and Y) are Z-scored to compare coefficients directly.
2.  **Model**: OLS Regression.
    *   $Y = \beta_0 + \beta_{peer} X_{peer} + \sum \beta_i X_i + \epsilon$

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import zscore
import os

# --- Paths ---
DATA_PATH = r"../../../../Data/2024data/TIGPS_W2_studentdata_ver5_cleaned_mental_common_only.csv"
RELATIONSHIP_PATH = r"../../relationship/Offline_Like.csv"

# Mental Health Scale Items
MH_COLS = [f"v55_{i}" for i in range(1, 15)]

# Top 20 Features (Manually selected based on correlation file)
TOP_20_FEATURES = [
    'v51', 'v59_5', 'v52_3', 'v57_4', 'v52_2', 
    'v57_3', 'v50', 'v57_2', 'v52', 'v57_1', 
    'v52_1', 'v57_5', 'v24_2', 'v8_08', 'v28_6', 
    'v39_2', 'v5_5', 'v8_05', 'v24_6', 'v35_2'
]

In [2]:
def load_and_process():
    print("1. Loading Student Data...")
    df = pd.read_csv(DATA_PATH, on_bad_lines='skip', engine='python')
    
    # 1. Calculate Dependent Variable (Y): Own Depression Score
    df['w2_own_score'] = df[MH_COLS].apply(pd.to_numeric, errors='coerce').sum(axis=1, min_count=1)
    
    # 2. Process Independent Variables (X): Top 20 Features
    for col in TOP_20_FEATURES:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(df[col].median())  # Fill NaNs with median
    
    # Filter valid rows
    valid_students = df.dropna(subset=['w2_own_score', 'student_id']).copy()
    print(f"   Valid Students: {len(valid_students)}")
    
    # --- Build Peer Score (X_peer) ---
    print("2. Calculating Peer Influence...")
    lookup = {}
    def clean_cls(v): return str(v).replace('.0', '').strip()
    
    # Create Score Lookup
    for idx, row in valid_students.iterrows():
        try:
            sch = int(pd.to_numeric(row['school_id'], errors='coerce'))
            cls = clean_cls(row['class'])
            seat = int(pd.to_numeric(row['v13'], errors='coerce'))
            lookup[(sch, cls, seat)] = (row['student_id'], row['w2_own_score'])
        except: continue
            
    # Process Edges
    edges = pd.read_csv(RELATIONSHIP_PATH)
    peer_data = {}
    
    for _, row in edges.iterrows():
        src_id = row['student_id']
        try:
            ts = int(pd.to_numeric(row['school_id'], errors='coerce'))
            tc = clean_cls(row['class'])
            tn = int(pd.to_numeric(row['nominated_seat_no'], errors='coerce'))
            
            if (ts, tc, tn) in lookup:
                tgt_id, tgt_score = lookup[(ts, tc, tn)]
                if src_id not in peer_data: peer_data[src_id] = []
                peer_data[src_id].append(tgt_score)
        except: continue
            
    # Average Peer Score
    peer_avg_list = []
    for sid, scores in peer_data.items():
        peer_avg_list.append({'student_id': sid, 'w2_peer_avg': np.mean(scores)})
    
    peer_df = pd.DataFrame(peer_avg_list)
    
    # 3. Merge Everything
    final_df = pd.merge(valid_students, peer_df, on='student_id', how='inner')
    print(f"   Final Sample Size: {len(final_df)}")
    
    return final_df

df = load_and_process()

1. Loading Student Data...
   Valid Students: 7175
2. Calculating Peer Influence...
   Final Sample Size: 6593


## 3. Standardization & Regression

In [3]:
# Z-Score Standardization for all variables
regression_vars = TOP_20_FEATURES + ['w2_peer_avg', 'w2_own_score']
df_std = df[regression_vars].apply(zscore)

# Rename columns for clarity in summary
# e.g., 'w2_peer_avg' -> 'Z_Peer'
rename_map = {col: f"Z_{col}" for col in TOP_20_FEATURES}
rename_map['w2_peer_avg'] = 'Z_Peer'
rename_map['w2_own_score'] = 'Z_Own_Depression'
df_std = df_std.rename(columns=rename_map)

# Define X and Y
X_cols = [f"Z_{col}" for col in TOP_20_FEATURES] + ['Z_Peer']
X = df_std[X_cols]
X = sm.add_constant(X) # Intercept
Y = df_std['Z_Own_Depression']

print("Running OLS Regression...")
model = sm.OLS(Y, X).fit()
print(model.summary())

Running OLS Regression...
                            OLS Regression Results                            
Dep. Variable:       Z_Own_Depression   R-squared:                       0.451
Model:                            OLS   Adj. R-squared:                  0.449
Method:                 Least Squares   F-statistic:                     257.2
Date:                Tue, 13 Jan 2026   Prob (F-statistic):               0.00
Time:                        18:13:58   Log-Likelihood:                -7377.2
No. Observations:                6593   AIC:                         1.480e+04
Df Residuals:                    6571   BIC:                         1.495e+04
Df Model:                          21                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -2.408e-16     

In [4]:
# Extract Key Results
peer_beta = model.params['Z_Peer']
peer_p = model.pvalues['Z_Peer']

print("\n--- Focus: Peer Influence in Context ---")
print(f"Peer Beta (Controlled): {peer_beta:.4f}")
print(f"Peer P-Value: {peer_p:.4e}")

if peer_p < 0.05:
    print("Conclusion: Even after controlling for 20 personal factors, Peer Influence is still SIGNIFICANT.")
else:
    print("Conclusion: Peer Influence becomes insignificant when personal factors are considered.")


--- Focus: Peer Influence in Context ---
Peer Beta (Controlled): 0.0897
Peer P-Value: 5.8894e-22
Conclusion: Even after controlling for 20 personal factors, Peer Influence is still SIGNIFICANT.
